In [3]:
from pathlib import Path
import pandas as pd

cwd = Path.cwd()

project_root = (
    cwd
    if (cwd / "data").exists()
    else cwd.parent
)

raw_dir = project_root / "data" / "raw"

df_2023 = pd.read_csv(
    raw_dir / "2023_24_sa_budget_raw.csv",
    dtype=str,
    keep_default_na=False
)

df_2024 = pd.read_csv(
    raw_dir / "2024_25_sa_budget_raw.csv",
    dtype=str,
    keep_default_na=False
)

df_2025 = pd.read_csv(
    raw_dir / "2025_26_sa_budget_raw.csv",
    dtype=str,
    keep_default_na=False
)

print("2023-24 rows:", len(df_2023))
print("2024-25 rows:", len(df_2024))
print("2025-26 rows:", len(df_2025))

2023-24 rows: 20
2024-25 rows: 20
2025-26 rows: 25


In [4]:
def get_program_names(df):
    names = (
        df["program_name_raw"]
        .astype(str)
        .str.replace("\n", " ", regex=False)
        .str.strip()
    )

    return sorted(
        name
        for name in names.unique()
        if name
        and name.lower() != "total"
    )


names_2023 = get_program_names(df_2023)
names_2024 = get_program_names(df_2024)
names_2025 = get_program_names(df_2025)

print("2023-24:", len(names_2023))
print("2024-25:", len(names_2024))
print("2025-26:", len(names_2025))

2023-24: 20
2024-25: 19
2025-26: 24


In [5]:
all_names = sorted(
    set(names_2023)
    | set(names_2024)
    | set(names_2025)
)

for name in all_names:
    print(name)

AANAPISI
ASC Book Fund
Accessibility, Community, & Opportunity
Associated Student Council
Bruce Mckenna Writing Center
Child Assist Program
Child Assist Program [combined in SSP]
Cultural Programming & Development (CAB)
Emergency Fund
Equity, Diversity, Inclusion, and Community
First Year Experience Peer Mentors
Food and Resource Pantry
Info Central
Information Central
Leadership & Orientation Training
Learning Support Network
M. Rosetta Hunter Art Gallary
M. Rosetta Hunter Art Gallery
Office Management
Parent Support Network
Phi Theta Kappa
SORC
SWAP
Seattle Collegian
Services & Activities Fees Committee
Student Engagement
Student Leadership Program
Student Organization Hub
Student Resource Support
Student Resource Support [combined in SSP]
Student Services
Student Support Program Supervisor
The Seattle Collegian
Umoja Scholars Program
Wood Technology Center


In [6]:
name_comparison = pd.DataFrame({
    "program_name_raw": all_names
})

name_comparison["in_2023_24_doc"] = (
    name_comparison["program_name_raw"]
    .isin(names_2023)
)

name_comparison["in_2024_25_doc"] = (
    name_comparison["program_name_raw"]
    .isin(names_2024)
)

name_comparison["in_2025_26_doc"] = (
    name_comparison["program_name_raw"]
    .isin(names_2025)
)

display(name_comparison)

,program_name_raw,in_2023_24_doc,in_2024_25_doc,in_2025_26_doc
0,AANAPISI,False,False,True
1,ASC Book Fund,True,True,True
2,"Accessibility, Community, & Opportunity",False,False,True
3,Associated Student Council,True,True,True
4,Bruce Mckenna Writing Center,True,False,False
5,Child Assist Program,False,True,False
6,Child Assist Program [combined in SSP],False,False,True
7,Cultural Programming & Development (CAB),True,True,True
8,Emergency Fund,True,True,True
9,"Equity, Diversity, Inclusion, and Community",True,True,False


In [7]:
crosswalk_records = [
    # Programs appearing consistently
    ["AANAPISI", "AANAPISI", "same", "First appears in FY2025-26 source."],
    ["ASC Book Fund", "ASC Book Fund", "same", ""],
    [
        "Accessibility, Community, & Opportunity",
        "Accessibility, Community, & Opportunity",
        "same",
        "Later program name used in the FY2025-26 source"
    ],
    ["Associated Student Council", "Associated Student Council", "same", ""],
    [
    "Bruce Mckenna Writing Center",   # raw PDF name: lowercase k
    "Bruce McKenna Writing Center",   # standardized name: uppercase K
    "spelling_variant",
    "Standardized capitalization of McKenna; appears in the earliest source only."
    ],
    [
        "Child Assist Program",
        "Child Assist Program",
        "same",
        ""
    ],
    [
        "Child Assist Program [combined in SSP]",
        "Child Assist Program",
        "merged_into_ssp",
        "FY2025-26 source states that the program was combined in SSP."
    ],
    [
        "Cultural Programming & Development (CAB)",
        "Cultural Programming & Development (CAB)",
        "same",
        ""
    ],
    ["Emergency Fund", "Emergency Fund", "same", ""],
    [
        "Equity, Diversity, Inclusion, and Community",
        "Accessibility, Community, & Opportunity",
        "renamed_or_reorganized",
        "FY2024-25 allocation of $9,076.80 is carried forward under Accessibility, Community, & Opportunity in the FY2025-26 source."
    ],
    [
        "First Year Experience Peer Mentors",
        "First Year Experience Peer Mentors",
        "same",
        "First appears in FY2025-26 source."
    ],
    [
        "Food and Resource Pantry",
        "Food and Resource Pantry",
        "same",
        "First appears in FY2025-26 source."
    ],

    # Naming changes
    [
        "Info Central",
        "Information Central",
        "renamed_or_relabelled",
        "Standardized to the later source name."
    ],
    [
        "Information Central",
        "Information Central",
        "same",
        ""
    ],

    ["Leadership & Orientation Training", "Leadership & Orientation Training", "same", ""],
    ["Learning Support Network", "Learning Support Network", "same", ""],

    # Typo correction
    [
        "M. Rosetta Hunter Art Gallary",
        "M. Rosetta Hunter Art Gallery",
        "spelling_variant",
        "Corrected source spelling variant: Gallary → Gallery."
    ],
    [
        "M. Rosetta Hunter Art Gallery",
        "M. Rosetta Hunter Art Gallery",
        "same",
        ""
    ],

    ["Office Management", "Office Management", "same", ""],
    [
        "Parent Support Network",
        "Parent Support Network",
        "historical_only",
        "Appears in the earliest source only."
    ],
    ["Phi Theta Kappa", "Phi Theta Kappa", "same", ""],

    # Club support program
    [
        "SORC",
        "Student Organization Hub",
        "renamed_or_relabelled",
        "Later source reports Student Organization Hub with the same prior-year allocation."
    ],

    # Student publication
    [
        "SWAP",
        "Seattle Collegian",
        "renamed_or_relabelled",
        "SWAP is Student Websites and Publications, which produces The Seattle Collegian."
    ],
    [
        "Seattle Collegian",
        "Seattle Collegian",
        "same",
        ""
    ],
    [
        "The Seattle Collegian",
        "Seattle Collegian",
        "naming_variant",
        "Standardized without leading 'The'."
    ],

    [
        "Services & Activities Fees Committee",
        "Services & Activities Fees Committee",
        "same",
        ""
    ],
    [
        "Student Engagement",
        "Student Engagement",
        "same",
        "First appears in the FY2024-25 source."
    ],
    ["Student Leadership Program", "Student Leadership Program", "same", ""],
    [
        "Student Organization Hub",
        "Student Organization Hub",
        "same",
        ""
    ],
    [
        "Student Resource Support",
        "Student Resource Support",
        "same",
        ""
    ],
    [
        "Student Resource Support [combined in SSP]",
        "Student Resource Support",
        "merged_into_ssp",
        "FY2025-26 source states that the program was combined in SSP."
    ],
    [
        "Student Services",
        "Student Services",
        "historical_only",
        "Appears in the earliest source only."
    ],
    [
        "Student Support Program Supervisor",
        "Student Support Program Supervisor",
        "same",
        "First appears in FY2025-26 source."
    ],
    [
        "Umoja Scholars Program",
        "Umoja Scholars Program",
        "same",
        "First appears in FY2025-26 source."
    ],
    ["Wood Technology Center", "Wood Technology Center", "same", ""],
]


program_crosswalk = pd.DataFrame(
    crosswalk_records,
    columns=[
        "program_name_raw",
        "program_name_standardized",
        "mapping_type",
        "notes",
    ],
)

display(program_crosswalk)

print("Crosswalk rows:", len(program_crosswalk))
print("Unique raw names:", program_crosswalk["program_name_raw"].nunique())

,program_name_raw,program_name_standardized,mapping_type,notes
0,AANAPISI,AANAPISI,same,First appears in FY2025-26 source.
1,ASC Book Fund,ASC Book Fund,same,
2,"Accessibility, Community, & Opportunity","Accessibility, Community, & Opportunity",same,Later program name used in the FY2025-26 source
3,Associated Student Council,Associated Student Council,same,
4,Bruce Mckenna Writing Center,Bruce McKenna Writing Center,spelling_variant,Standardized capitalization of McKenna; appear...
5,Child Assist Program,Child Assist Program,same,
6,Child Assist Program [combined in SSP],Child Assist Program,merged_into_ssp,FY2025-26 source states that the program was c...
7,Cultural Programming & Development (CAB),Cultural Programming & Development (CAB),same,
8,Emergency Fund,Emergency Fund,same,
9,"Equity, Diversity, Inclusion, and Community","Accessibility, Community, & Opportunity",renamed_or_reorganized,"FY2024-25 allocation of $9,076.80 is carried f..."


Crosswalk rows: 35
Unique raw names: 35


In [8]:
crosswalk_names = set(
    program_crosswalk["program_name_raw"]
)

observed_names = set(all_names)

missing_from_crosswalk = sorted(
    observed_names - crosswalk_names
)

extra_in_crosswalk = sorted(
    crosswalk_names - observed_names
)

print("Missing from crosswalk:")
print(missing_from_crosswalk)

print("\nExtra in crosswalk:")
print(extra_in_crosswalk)

Missing from crosswalk:
[]

Extra in crosswalk:
[]


In [9]:
manual_dir = project_root / "data" / "manual"

crosswalk_path = manual_dir / "program_crosswalk.csv"

program_crosswalk.to_csv(
    crosswalk_path,
    index=False
)

print("Saved to:")
print(crosswalk_path)

Saved to:
c:\GitHub Projects\seattle-central-sa-budget-dashboard\data\manual\program_crosswalk.csv


In [10]:
check_crosswalk = pd.read_csv(
    crosswalk_path,
    dtype=str,
    keep_default_na=False
)

print("Rows:", len(check_crosswalk))
print("Columns:", len(check_crosswalk.columns))

display(check_crosswalk.head())
display(check_crosswalk.tail())

Rows: 35
Columns: 4


,program_name_raw,program_name_standardized,mapping_type,notes
0,AANAPISI,AANAPISI,same,First appears in FY2025-26 source.
1,ASC Book Fund,ASC Book Fund,same,
2,"Accessibility, Community, & Opportunity","Accessibility, Community, & Opportunity",same,Later program name used in the FY2025-26 source
3,Associated Student Council,Associated Student Council,same,
4,Bruce Mckenna Writing Center,Bruce McKenna Writing Center,spelling_variant,Standardized capitalization of McKenna; appear...


,program_name_raw,program_name_standardized,mapping_type,notes
30,Student Resource Support [combined in SSP],Student Resource Support,merged_into_ssp,FY2025-26 source states that the program was c...
31,Student Services,Student Services,historical_only,Appears in the earliest source only.
32,Student Support Program Supervisor,Student Support Program Supervisor,same,First appears in FY2025-26 source.
33,Umoja Scholars Program,Umoja Scholars Program,same,First appears in FY2025-26 source.
34,Wood Technology Center,Wood Technology Center,same,


In [11]:
import numpy as np


def normalize_program_name(value):
    if value is None:
        return ""

    return (
        str(value)
        .replace("\n", " ")
        .strip()
    )


def parse_currency(value):
    if value is None:
        return np.nan

    value = str(value).strip()

    # Preserve non-reported / not-applicable values as missing
    if value in {
        "",
        "-",
        "--",
        "—",
        "–",
    }:
        return np.nan

    value = (
        value
        .replace("$", "")
        .replace(",", "")
    )

    return float(value)

In [12]:
clean_2023 = df_2023.copy()

# Remove Total if one ever appears
clean_2023["program_name_clean"] = (
    clean_2023["program_name_raw"]
    .apply(normalize_program_name)
)

clean_2023 = clean_2023[
    clean_2023["program_name_clean"]
    .str.lower()
    .ne("total")
].copy()

# Add standardized program names
clean_2023 = clean_2023.merge(
    program_crosswalk[
        [
            "program_name_raw",
            "program_name_standardized",
            "mapping_type",
        ]
    ],
    left_on="program_name_clean",
    right_on="program_name_raw",
    how="left",
    suffixes=("", "_crosswalk"),
)

# Convert currency fields to numeric
clean_2023["allocation_2022_23"] = (
    clean_2023["allocation_2022_23_raw"]
    .apply(parse_currency)
)

clean_2023["request_2023_24"] = (
    clean_2023["request_2023_24_raw"]
    .apply(parse_currency)
)

clean_2023["allocation_2023_24"] = (
    clean_2023["allocation_2023_24_raw"]
    .apply(parse_currency)
)

display(
    clean_2023[
        [
            "program_name_clean",
            "program_name_standardized",
            "allocation_2022_23",
            "request_2023_24",
            "allocation_2023_24",
        ]
    ]
)

,program_name_clean,program_name_standardized,allocation_2022_23,request_2023_24,allocation_2023_24
0,ASC Book Fund,ASC Book Fund,3000.00,3000.00,2000.00
1,Associated Student Council,Associated Student Council,56953.00,64539.00,64539.00
2,Bruce Mckenna Writing Center,Bruce McKenna Writing Center,56039.01,0.00,0.00
3,Cultural Programming & Development (CAB),Cultural Programming & Development (CAB),116767.00,116767.00,116767.00
4,Emergency Fund,Emergency Fund,25000.00,50000.00,25000.00
5,"Equity, Diversity, Inclusion, and Community","Accessibility, Community, & Opportunity",11500.00,11500.00,9000.00
6,Info Central,Information Central,157950.76,169521.00,157950.00
7,Leadership & Orientation Training,Leadership & Orientation Training,12600.00,8500.00,8500.00
8,Learning Support Network,Learning Support Network,419159.00,540680.00,470430.25
9,M. Rosetta Hunter Art Gallary,M. Rosetta Hunter Art Gallery,57513.12,62872.00,57513.12


In [13]:
print(
    "Missing standardized names:",
    clean_2023["program_name_standardized"]
    .isna()
    .sum()
)

print()

print("Numeric totals:")
print(
    "2022-23 Allocation:",
    f"${clean_2023['allocation_2022_23'].sum():,.2f}"
)

print(
    "2023-24 Request:",
    f"${clean_2023['request_2023_24'].sum():,.2f}"
)

print(
    "2023-24 Allocation:",
    f"${clean_2023['allocation_2023_24'].sum():,.2f}"
)

Missing standardized names: 0

Numeric totals:
2022-23 Allocation: $1,500,000.00
2023-24 Request: $1,633,326.63
2023-24 Allocation: $1,500,000.00


In [14]:
display(
    clean_2023[
        clean_2023[
            [
                "allocation_2022_23",
                "request_2023_24",
                "allocation_2023_24",
            ]
        ]
        .isna()
        .any(axis=1)
    ][
        [
            "program_name_standardized",
            "allocation_2022_23_raw",
            "request_2023_24_raw",
            "allocation_2023_24_raw",
            "allocation_2022_23",
            "request_2023_24",
            "allocation_2023_24",
        ]
    ]
)

,program_name_standardized,allocation_2022_23_raw,request_2023_24_raw,allocation_2023_24_raw,allocation_2022_23,request_2023_24,allocation_2023_24
13,Services & Activities Fees Committee,-,"$5,416.00","$5,416.00",NaN,5416.0,5416.0
19,Wood Technology Center,"$12,000.00",-,"$4,000.00",12000.0,NaN,4000.0


In [15]:
clean_2024 = df_2024.copy()

clean_2024["program_name_clean"] = (
    clean_2024["program_name_raw"]
    .apply(normalize_program_name)
)

# Remove Total row
clean_2024 = clean_2024[
    clean_2024["program_name_clean"]
    .str.lower()
    .ne("total")
].copy()

# Standardize program names
clean_2024 = clean_2024.merge(
    program_crosswalk[
        [
            "program_name_raw",
            "program_name_standardized",
            "mapping_type",
        ]
    ],
    left_on="program_name_clean",
    right_on="program_name_raw",
    how="left",
    suffixes=("", "_crosswalk"),
)

# Convert currency columns
clean_2024["allocation_2023_24"] = (
    clean_2024["allocation_2023_24_raw"]
    .apply(parse_currency)
)

clean_2024["request_2024_25"] = (
    clean_2024["request_2024_25_raw"]
    .apply(parse_currency)
)

clean_2024["allocation_2024_25"] = (
    clean_2024["allocation_2024_25_raw"]
    .apply(parse_currency)
)

In [16]:
clean_2025 = df_2025.copy()

clean_2025["program_name_clean"] = (
    clean_2025["program_name_raw"]
    .apply(normalize_program_name)
)

# Remove Total row
clean_2025 = clean_2025[
    clean_2025["program_name_clean"]
    .str.lower()
    .ne("total")
].copy()

# Standardize program names
clean_2025 = clean_2025.merge(
    program_crosswalk[
        [
            "program_name_raw",
            "program_name_standardized",
            "mapping_type",
        ]
    ],
    left_on="program_name_clean",
    right_on="program_name_raw",
    how="left",
    suffixes=("", "_crosswalk"),
)

# Convert currency columns
clean_2025["allocation_2023_24"] = (
    clean_2025["allocation_2023_24_raw"]
    .apply(parse_currency)
)

clean_2025["allocation_2024_25"] = (
    clean_2025["allocation_2024_25_raw"]
    .apply(parse_currency)
)

clean_2025["request_2025_26"] = (
    clean_2025["request_2025_26_raw"]
    .apply(parse_currency)
)

clean_2025["allocation_2025_26"] = (
    clean_2025["allocation_2025_26_raw"]
    .apply(parse_currency)
)

In [17]:
print("PDF #2 missing standardized names:")
print(
    clean_2024["program_name_standardized"]
    .isna()
    .sum()
)

print("\nPDF #2 totals:")
print(
    f"2023-24 Allocation: "
    f"${clean_2024['allocation_2023_24'].sum():,.2f}"
)
print(
    f"2024-25 Request: "
    f"${clean_2024['request_2024_25'].sum():,.2f}"
)
print(
    f"2024-25 Allocation: "
    f"${clean_2024['allocation_2024_25'].sum():,.2f}"
)


print("\nPDF #3 missing standardized names:")
print(
    clean_2025["program_name_standardized"]
    .isna()
    .sum()
)

print("\nPDF #3 totals:")
print(
    f"2023-24 Allocation: "
    f"${clean_2025['allocation_2023_24'].sum():,.2f}"
)
print(
    f"2024-25 Allocation: "
    f"${clean_2025['allocation_2024_25'].sum():,.2f}"
)
print(
    f"2025-26 Request: "
    f"${clean_2025['request_2025_26'].sum():,.2f}"
)
print(
    f"2025-26 Allocation: "
    f"${clean_2025['allocation_2025_26'].sum():,.2f}"
)

PDF #2 missing standardized names:
0

PDF #2 totals:
2023-24 Allocation: $1,513,171.18
2024-25 Request: $1,502,598.56
2024-25 Allocation: $1,733,008.51

PDF #3 missing standardized names:
0

PDF #3 totals:
2023-24 Allocation: $1,513,171.18
2024-25 Allocation: $1,733,008.51
2025-26 Request: $2,046,910.19
2025-26 Allocation: $1,801,888.48


In [18]:
compare_2023_pdf1 = (
    clean_2023[
        [
            "program_name_standardized",
            "allocation_2023_24",
        ]
    ]
    .rename(
        columns={
            "allocation_2023_24":
                "allocation_2023_24_pdf1"
        }
    )
)

compare_2023_pdf2 = (
    clean_2024[
        [
            "program_name_standardized",
            "allocation_2023_24",
        ]
    ]
    .rename(
        columns={
            "allocation_2023_24":
                "allocation_2023_24_pdf2"
        }
    )
)

compare_2023_pdf3 = (
    clean_2025[
        [
            "program_name_standardized",
            "allocation_2023_24",
        ]
    ]
    .rename(
        columns={
            "allocation_2023_24":
                "allocation_2023_24_pdf3"
        }
    )
)

In [19]:
comparison_2023_24 = (
    compare_2023_pdf1
    .merge(
        compare_2023_pdf2,
        on="program_name_standardized",
        how="outer",
    )
    .merge(
        compare_2023_pdf3,
        on="program_name_standardized",
        how="outer",
    )
)

comparison_2023_24[
    "difference_pdf2_vs_pdf1"
] = (
    comparison_2023_24[
        "allocation_2023_24_pdf2"
    ]
    -
    comparison_2023_24[
        "allocation_2023_24_pdf1"
    ]
)

comparison_2023_24[
    "difference_pdf3_vs_pdf2"
] = (
    comparison_2023_24[
        "allocation_2023_24_pdf3"
    ]
    -
    comparison_2023_24[
        "allocation_2023_24_pdf2"
    ]
)

display(
    comparison_2023_24.sort_values(
        "program_name_standardized"
    )
)

,program_name_standardized,allocation_2023_24_pdf1,allocation_2023_24_pdf2,allocation_2023_24_pdf3,difference_pdf2_vs_pdf1,difference_pdf3_vs_pdf2
0,AANAPISI,NaN,NaN,0.00,NaN,NaN
1,ASC Book Fund,2000.00,2000.00,2000.00,0.00,0.0
2,"Accessibility, Community, & Opportunity",9000.00,9000.00,9000.00,0.00,0.0
3,Associated Student Council,64539.00,64539.00,64539.00,0.00,0.0
4,Bruce McKenna Writing Center,0.00,NaN,NaN,NaN,NaN
5,Child Assist Program,NaN,20000.00,20000.00,NaN,0.0
6,Cultural Programming & Development (CAB),116767.00,116767.00,116767.00,0.00,0.0
7,Emergency Fund,25000.00,25000.00,25000.00,0.00,0.0
8,First Year Experience Peer Mentors,NaN,NaN,0.00,NaN,NaN
9,Food and Resource Pantry,NaN,NaN,0.00,NaN,NaN


In [20]:
changed_2023_24 = comparison_2023_24[
    (
        comparison_2023_24[
            "difference_pdf2_vs_pdf1"
        ].fillna(0).abs() > 0.01
    )
    |
    (
        comparison_2023_24[
            "difference_pdf3_vs_pdf2"
        ].fillna(0).abs() > 0.01
    )
    |
    comparison_2023_24[
        [
            "allocation_2023_24_pdf1",
            "allocation_2023_24_pdf2",
            "allocation_2023_24_pdf3",
        ]
    ].isna().any(axis=1)
]

display(changed_2023_24)

,program_name_standardized,allocation_2023_24_pdf1,allocation_2023_24_pdf2,allocation_2023_24_pdf3,difference_pdf2_vs_pdf1,difference_pdf3_vs_pdf2
0,AANAPISI,NaN,NaN,0.0,NaN,NaN
4,Bruce McKenna Writing Center,0.00,NaN,NaN,NaN,NaN
5,Child Assist Program,NaN,20000.0,20000.0,NaN,0.0
8,First Year Experience Peer Mentors,NaN,NaN,0.0,NaN,NaN
9,Food and Resource Pantry,NaN,NaN,0.0,NaN,NaN
11,Leadership & Orientation Training,8500.00,16384.0,16384.0,7884.00,0.0
13,M. Rosetta Hunter Art Gallery,57513.12,57513.0,57513.0,-0.12,0.0
15,Parent Support Network,20000.00,NaN,NaN,NaN,NaN
18,Services & Activities Fees Committee,5416.00,7500.0,7500.0,2084.00,0.0
19,Student Engagement,NaN,0.0,0.0,NaN,0.0


In [21]:
actual_changes_2023_24 = comparison_2023_24[
    comparison_2023_24[
        "allocation_2023_24_pdf1"
    ].notna()
    &
    comparison_2023_24[
        "allocation_2023_24_pdf2"
    ].notna()
    &
    (
        comparison_2023_24[
            "difference_pdf2_vs_pdf1"
        ].abs() > 0.01
    )
].copy()

display(
    actual_changes_2023_24[
        [
            "program_name_standardized",
            "allocation_2023_24_pdf1",
            "allocation_2023_24_pdf2",
            "difference_pdf2_vs_pdf1",
        ]
    ]
)

print(
    "Net restatement:",
    f"${actual_changes_2023_24['difference_pdf2_vs_pdf1'].sum():,.2f}"
)

,program_name_standardized,allocation_2023_24_pdf1,allocation_2023_24_pdf2,difference_pdf2_vs_pdf1
11,Leadership & Orientation Training,8500.00,16384.0,7884.00
13,M. Rosetta Hunter Art Gallery,57513.12,57513.0,-0.12
18,Services & Activities Fees Committee,5416.00,7500.0,2084.00
22,Student Resource Support,136000.00,139203.3,3203.30


Net restatement: $13,171.18


In [22]:
compare_2024_pdf2 = (
    clean_2024[
        [
            "program_name_standardized",
            "allocation_2024_25",
        ]
    ]
    .rename(
        columns={
            "allocation_2024_25":
                "allocation_2024_25_pdf2"
        }
    )
)

compare_2024_pdf3 = (
    clean_2025[
        [
            "program_name_standardized",
            "allocation_2024_25",
        ]
    ]
    .rename(
        columns={
            "allocation_2024_25":
                "allocation_2024_25_pdf3"
        }
    )
)

comparison_2024_25 = (
    compare_2024_pdf2
    .merge(
        compare_2024_pdf3,
        on="program_name_standardized",
        how="outer",
    )
)

comparison_2024_25["difference"] = (
    comparison_2024_25["allocation_2024_25_pdf3"]
    -
    comparison_2024_25["allocation_2024_25_pdf2"]
)

changed_2024_25 = comparison_2024_25[
    (
        comparison_2024_25["difference"]
        .fillna(0)
        .abs() > 0.01
    )
    |
    comparison_2024_25[
        [
            "allocation_2024_25_pdf2",
            "allocation_2024_25_pdf3",
        ]
    ].isna().any(axis=1)
]

display(changed_2024_25)

,program_name_standardized,allocation_2024_25_pdf2,allocation_2024_25_pdf3,difference
0,AANAPISI,NaN,0.0,NaN
7,First Year Experience Peer Mentors,NaN,0.0,NaN
8,Food and Resource Pantry,NaN,0.0,NaN
21,Student Support Program Supervisor,NaN,0.0,NaN
22,Umoja Scholars Program,NaN,0.0,NaN


In [23]:
# ============================================================
# FINAL CLEANING & STANDARDIZATION VALIDATION
# ============================================================

import math

print("Running validation checks...\n")

# ------------------------------------------------------------
# 1. Crosswalk coverage
# ------------------------------------------------------------

crosswalk_names = set(program_crosswalk["program_name_raw"])
observed_names = set(all_names)

assert observed_names == crosswalk_names, (
    "Crosswalk coverage failed.\n"
    f"Missing: {sorted(observed_names - crosswalk_names)}\n"
    f"Extra: {sorted(crosswalk_names - observed_names)}"
)

assert program_crosswalk["program_name_raw"].nunique() == 35
assert len(program_crosswalk) == 35

print("✓ Crosswalk covers all 35 raw program names")


# ------------------------------------------------------------
# 2. Standardized-name coverage
# ------------------------------------------------------------

for label, df in {
    "2023-24": clean_2023,
    "2024-25": clean_2024,
    "2025-26": clean_2025,
}.items():

    missing = (
        df["program_name_standardized"]
        .isna()
        .sum()
    )

    assert missing == 0, (
        f"{label}: {missing} programs are missing "
        "standardized names."
    )

print("✓ No missing standardized program names")


# ------------------------------------------------------------
# 3. Expected program-row counts
# ------------------------------------------------------------

expected_rows = {
    "2023-24": 20,
    "2024-25": 19,
    "2025-26": 24,
}

actual_rows = {
    "2023-24": len(clean_2023),
    "2024-25": len(clean_2024),
    "2025-26": len(clean_2025),
}

assert actual_rows == expected_rows, (
    f"Unexpected row counts: {actual_rows}"
)

print("✓ Program row counts match source documents")


# ------------------------------------------------------------
# 4. No duplicate standardized programs within a source
# ------------------------------------------------------------

for label, df in {
    "2023-24": clean_2023,
    "2024-25": clean_2024,
    "2025-26": clean_2025,
}.items():

    duplicates = df[
        df["program_name_standardized"]
        .duplicated(keep=False)
    ]

    assert duplicates.empty, (
        f"{label}: duplicate standardized programs found:\n"
        f"{duplicates['program_name_standardized'].tolist()}"
    )

print("✓ No duplicate standardized programs within a source")


# ------------------------------------------------------------
# 5. Expected financial totals
# ------------------------------------------------------------

expected_totals = {
    "2022-23 allocation": (
        clean_2023["allocation_2022_23"].sum(),
        1_500_000.00,
    ),

    "2023-24 original allocation": (
        clean_2023["allocation_2023_24"].sum(),
        1_500_000.00,
    ),

    "2023-24 restated allocation PDF2": (
        clean_2024["allocation_2023_24"].sum(),
        1_513_171.18,
    ),

    "2023-24 restated allocation PDF3": (
        clean_2025["allocation_2023_24"].sum(),
        1_513_171.18,
    ),

    "2024-25 request": (
        clean_2024["request_2024_25"].sum(),
        1_502_598.56,
    ),

    "2024-25 allocation PDF2": (
        clean_2024["allocation_2024_25"].sum(),
        1_733_008.51,
    ),

    "2024-25 allocation PDF3": (
        clean_2025["allocation_2024_25"].sum(),
        1_733_008.51,
    ),

    "2025-26 request": (
        clean_2025["request_2025_26"].sum(),
        2_046_910.19,
    ),

    "2025-26 allocation": (
        clean_2025["allocation_2025_26"].sum(),
        1_801_888.48,
    ),
}

for label, (actual, expected) in expected_totals.items():

    assert math.isclose(
        actual,
        expected,
        abs_tol=0.01,
    ), (
        f"{label} failed: "
        f"expected ${expected:,.2f}, "
        f"got ${actual:,.2f}"
    )

print("✓ All financial totals reconcile")


# ------------------------------------------------------------
# 6. Confirm historical restatement
# ------------------------------------------------------------

restatement = (
    clean_2024["allocation_2023_24"].sum()
    -
    clean_2023["allocation_2023_24"].sum()
)

assert math.isclose(
    restatement,
    13_171.18,
    abs_tol=0.01,
)

print(
    "✓ FY2023-24 restatement confirmed: "
    f"${restatement:,.2f}"
)


# ------------------------------------------------------------
# 7. Confirm current crosswalk was saved to disk
# ------------------------------------------------------------

saved_crosswalk = pd.read_csv(
    crosswalk_path,
    dtype=str,
    keep_default_na=False,
)

current_check = (
    program_crosswalk
    .fillna("")
    .reset_index(drop=True)
)

saved_check = (
    saved_crosswalk
    .fillna("")
    .reset_index(drop=True)
)

assert current_check.equals(saved_check), (
    "Saved program_crosswalk.csv does not match "
    "the current notebook version."
)

print("✓ Saved program_crosswalk.csv matches notebook")


print("\n" + "=" * 55)
print("ALL VALIDATION CHECKS PASSED ✓")
print("=" * 55)

Running validation checks...

✓ Crosswalk covers all 35 raw program names
✓ No missing standardized program names
✓ Program row counts match source documents
✓ No duplicate standardized programs within a source
✓ All financial totals reconcile
✓ FY2023-24 restatement confirmed: $13,171.18
✓ Saved program_crosswalk.csv matches notebook

ALL VALIDATION CHECKS PASSED ✓


In [24]:
def build_long_rows(
    df,
    metric_specs,
):
    """
    Convert one cleaned source table from wide format
    into long format while preserving source metadata
    and original raw values.
    """

    pieces = []

    for spec in metric_specs:
        piece = df[
            [
                "source_document",
                "source_page",
                "source_url",
                "program_name_raw",
                "program_name_standardized",
                "mapping_type",
            ]
        ].copy()

        piece["fiscal_year"] = spec["fiscal_year"]
        piece["metric_type"] = spec["metric_type"]

        piece["amount"] = df[
            spec["numeric_column"]
        ]

        piece["raw_value"] = df[
            spec["raw_column"]
        ]

        piece["metric_context"] = spec.get(
            "metric_context",
            ""
        )

        pieces.append(piece)

    return pd.concat(
        pieces,
        ignore_index=True,
    )

In [25]:
reported_2023 = build_long_rows(
    clean_2023,
    [
        {
            "fiscal_year": "2022-23",
            "metric_type": "allocation",
            "numeric_column": "allocation_2022_23",
            "raw_column": "allocation_2022_23_raw",
        },
        {
            "fiscal_year": "2023-24",
            "metric_type": "request",
            "numeric_column": "request_2023_24",
            "raw_column": "request_2023_24_raw",
        },
        {
            "fiscal_year": "2023-24",
            "metric_type": "allocation",
            "numeric_column": "allocation_2023_24",
            "raw_column": "allocation_2023_24_raw",
        },
    ],
)

In [26]:
reported_2024 = build_long_rows(
    clean_2024,
    [
        {
            "fiscal_year": "2023-24",
            "metric_type": "allocation",
            "numeric_column": "allocation_2023_24",
            "raw_column": "allocation_2023_24_raw",
        },
        {
            "fiscal_year": "2024-25",
            "metric_type": "request",
            "numeric_column": "request_2024_25",
            "raw_column": "request_2024_25_raw",
            "metric_context": "biennial_adjustment_request",
        },
        {
            "fiscal_year": "2024-25",
            "metric_type": "allocation",
            "numeric_column": "allocation_2024_25",
            "raw_column": "allocation_2024_25_raw",
        },
    ],
)

In [27]:
reported_2025 = build_long_rows(
    clean_2025,
    [
        {
            "fiscal_year": "2023-24",
            "metric_type": "allocation",
            "numeric_column": "allocation_2023_24",
            "raw_column": "allocation_2023_24_raw",
        },
        {
            "fiscal_year": "2024-25",
            "metric_type": "allocation",
            "numeric_column": "allocation_2024_25",
            "raw_column": "allocation_2024_25_raw",
        },
        {
            "fiscal_year": "2025-26",
            "metric_type": "request",
            "numeric_column": "request_2025_26",
            "raw_column": "request_2025_26_raw",
        },
        {
            "fiscal_year": "2025-26",
            "metric_type": "allocation",
            "numeric_column": "allocation_2025_26",
            "raw_column": "allocation_2025_26_raw",
        },
    ],
)

In [28]:
sa_budget_reported_long = pd.concat(
    [
        reported_2023,
        reported_2024,
        reported_2025,
    ],
    ignore_index=True,
)

sa_budget_reported_long = (
    sa_budget_reported_long
    .sort_values(
        [
            "fiscal_year",
            "metric_type",
            "program_name_standardized",
            "source_document",
        ]
    )
    .reset_index(drop=True)
)

display(sa_budget_reported_long.head(10))

print(
    "Rows:",
    len(sa_budget_reported_long)
)

,source_document,source_page,source_url,program_name_raw,program_name_standardized,mapping_type,fiscal_year,metric_type,amount,raw_value,metric_context
0,2023-24_sa_fee_memo.pdf,3,https://studentleadership.seattlecentral.edu/s...,ASC Book Fund,ASC Book Fund,same,2022-23,allocation,3000.00,"$3,000.00",
1,2023-24_sa_fee_memo.pdf,3,https://studentleadership.seattlecentral.edu/s...,"Equity, Diversity, Inclusion, and Community","Accessibility, Community, & Opportunity",renamed_or_reorganized,2022-23,allocation,11500.00,"$11,500.00",
2,2023-24_sa_fee_memo.pdf,3,https://studentleadership.seattlecentral.edu/s...,Associated Student Council,Associated Student Council,same,2022-23,allocation,56953.00,"$56,953.00",
3,2023-24_sa_fee_memo.pdf,3,https://studentleadership.seattlecentral.edu/s...,Bruce Mckenna Writing Center,Bruce McKenna Writing Center,spelling_variant,2022-23,allocation,56039.01,"$56,039.01",
4,2023-24_sa_fee_memo.pdf,3,https://studentleadership.seattlecentral.edu/s...,Cultural Programming & Development (CAB),Cultural Programming & Development (CAB),same,2022-23,allocation,116767.00,"$116,767.00",
5,2023-24_sa_fee_memo.pdf,3,https://studentleadership.seattlecentral.edu/s...,Emergency Fund,Emergency Fund,same,2022-23,allocation,25000.00,"$25,000.00",
6,2023-24_sa_fee_memo.pdf,3,https://studentleadership.seattlecentral.edu/s...,Info Central,Information Central,renamed_or_relabelled,2022-23,allocation,157950.76,"$157,950.76",
7,2023-24_sa_fee_memo.pdf,3,https://studentleadership.seattlecentral.edu/s...,Leadership & Orientation Training,Leadership & Orientation Training,same,2022-23,allocation,12600.00,"$12,600.00",
8,2023-24_sa_fee_memo.pdf,3,https://studentleadership.seattlecentral.edu/s...,Learning Support Network,Learning Support Network,same,2022-23,allocation,419159.00,"$419,159.00",
9,2023-24_sa_fee_memo.pdf,3,https://studentleadership.seattlecentral.edu/s...,M. Rosetta Hunter Art Gallary,M. Rosetta Hunter Art Gallery,spelling_variant,2022-23,allocation,57513.12,"$57,513.12",


Rows: 213


In [29]:
display(
    sa_budget_reported_long[
        [
            "source_document",
            "source_page",
            "program_name_standardized",
            "fiscal_year",
            "metric_type",
            "amount",
            "raw_value",
            "metric_context",
        ]
    ].head(20)
)

,source_document,source_page,program_name_standardized,fiscal_year,metric_type,amount,raw_value,metric_context
0,2023-24_sa_fee_memo.pdf,3,ASC Book Fund,2022-23,allocation,3000.00,"$3,000.00",
1,2023-24_sa_fee_memo.pdf,3,"Accessibility, Community, & Opportunity",2022-23,allocation,11500.00,"$11,500.00",
2,2023-24_sa_fee_memo.pdf,3,Associated Student Council,2022-23,allocation,56953.00,"$56,953.00",
3,2023-24_sa_fee_memo.pdf,3,Bruce McKenna Writing Center,2022-23,allocation,56039.01,"$56,039.01",
4,2023-24_sa_fee_memo.pdf,3,Cultural Programming & Development (CAB),2022-23,allocation,116767.00,"$116,767.00",
5,2023-24_sa_fee_memo.pdf,3,Emergency Fund,2022-23,allocation,25000.00,"$25,000.00",
6,2023-24_sa_fee_memo.pdf,3,Information Central,2022-23,allocation,157950.76,"$157,950.76",
7,2023-24_sa_fee_memo.pdf,3,Leadership & Orientation Training,2022-23,allocation,12600.00,"$12,600.00",
8,2023-24_sa_fee_memo.pdf,3,Learning Support Network,2022-23,allocation,419159.00,"$419,159.00",
9,2023-24_sa_fee_memo.pdf,3,M. Rosetta Hunter Art Gallery,2022-23,allocation,57513.12,"$57,513.12",


In [30]:
print("Total rows:", len(sa_budget_reported_long))

print("\nRows by source:")
print(
    sa_budget_reported_long[
        "source_document"
    ].value_counts()
)

print("\nRows by fiscal year and metric:")
display(
    sa_budget_reported_long
    .groupby(
        [
            "fiscal_year",
            "metric_type",
        ],
        dropna=False,
    )
    .size()
    .reset_index(name="rows")
)

Total rows: 213

Rows by source:
source_document
2025-26_sa_budget_summary.pdf    96
2023-24_sa_fee_memo.pdf          60
2024-25_sa_fee_memo.pdf          57
Name: count, dtype: int64

Rows by fiscal year and metric:


,fiscal_year,metric_type,rows
0,2022-23,allocation,20
1,2023-24,allocation,63
2,2023-24,request,20
3,2024-25,allocation,43
4,2024-25,request,19
5,2025-26,allocation,24
6,2025-26,request,24


In [31]:
processed_dir = (
    project_root
    / "data"
    / "processed"
)

reported_long_path = (
    processed_dir
    / "sa_budget_reported_long.csv"
)

sa_budget_reported_long.to_csv(
    reported_long_path,
    index=False,
)

print("Saved:")
print(reported_long_path)

Saved:
c:\GitHub Projects\seattle-central-sa-budget-dashboard\data\processed\sa_budget_reported_long.csv


In [32]:
check_reported_long = pd.read_csv(
    reported_long_path,
    dtype={
        "source_page": str,
    },
)

print(
    "Saved rows:",
    len(check_reported_long)
)

print(
    "Saved columns:",
    len(check_reported_long.columns)
)

Saved rows: 213
Saved columns: 11


In [33]:
canonical_rules = [
    {
        "fiscal_year": "2022-23",
        "metric_type": "allocation",
        "source_document": "2023-24_sa_fee_memo.pdf",
    },
    {
        "fiscal_year": "2023-24",
        "metric_type": "request",
        "source_document": "2023-24_sa_fee_memo.pdf",
    },
    {
        "fiscal_year": "2023-24",
        "metric_type": "allocation",
        "source_document": "2025-26_sa_budget_summary.pdf",
    },
    {
        "fiscal_year": "2024-25",
        "metric_type": "request",
        "source_document": "2024-25_sa_fee_memo.pdf",
    },
    {
        "fiscal_year": "2024-25",
        "metric_type": "allocation",
        "source_document": "2025-26_sa_budget_summary.pdf",
    },
    {
        "fiscal_year": "2025-26",
        "metric_type": "request",
        "source_document": "2025-26_sa_budget_summary.pdf",
    },
    {
        "fiscal_year": "2025-26",
        "metric_type": "allocation",
        "source_document": "2025-26_sa_budget_summary.pdf",
    },
]

In [34]:
canonical_parts = []

for rule in canonical_rules:
    part = sa_budget_reported_long[
        (
            sa_budget_reported_long["fiscal_year"]
            == rule["fiscal_year"]
        )
        &
        (
            sa_budget_reported_long["metric_type"]
            == rule["metric_type"]
        )
        &
        (
            sa_budget_reported_long["source_document"]
            == rule["source_document"]
        )
    ].copy()

    canonical_parts.append(part)


sa_budget_canonical = pd.concat(
    canonical_parts,
    ignore_index=True,
)

sa_budget_canonical = (
    sa_budget_canonical
    .sort_values(
        [
            "fiscal_year",
            "metric_type",
            "program_name_standardized",
        ]
    )
    .reset_index(drop=True)
)

print(
    "Canonical rows:",
    len(sa_budget_canonical)
)

display(
    sa_budget_canonical.head(10)
)

Canonical rows: 155


,source_document,source_page,source_url,program_name_raw,program_name_standardized,mapping_type,fiscal_year,metric_type,amount,raw_value,metric_context
0,2023-24_sa_fee_memo.pdf,3,https://studentleadership.seattlecentral.edu/s...,ASC Book Fund,ASC Book Fund,same,2022-23,allocation,3000.00,"$3,000.00",
1,2023-24_sa_fee_memo.pdf,3,https://studentleadership.seattlecentral.edu/s...,"Equity, Diversity, Inclusion, and Community","Accessibility, Community, & Opportunity",renamed_or_reorganized,2022-23,allocation,11500.00,"$11,500.00",
2,2023-24_sa_fee_memo.pdf,3,https://studentleadership.seattlecentral.edu/s...,Associated Student Council,Associated Student Council,same,2022-23,allocation,56953.00,"$56,953.00",
3,2023-24_sa_fee_memo.pdf,3,https://studentleadership.seattlecentral.edu/s...,Bruce Mckenna Writing Center,Bruce McKenna Writing Center,spelling_variant,2022-23,allocation,56039.01,"$56,039.01",
4,2023-24_sa_fee_memo.pdf,3,https://studentleadership.seattlecentral.edu/s...,Cultural Programming & Development (CAB),Cultural Programming & Development (CAB),same,2022-23,allocation,116767.00,"$116,767.00",
5,2023-24_sa_fee_memo.pdf,3,https://studentleadership.seattlecentral.edu/s...,Emergency Fund,Emergency Fund,same,2022-23,allocation,25000.00,"$25,000.00",
6,2023-24_sa_fee_memo.pdf,3,https://studentleadership.seattlecentral.edu/s...,Info Central,Information Central,renamed_or_relabelled,2022-23,allocation,157950.76,"$157,950.76",
7,2023-24_sa_fee_memo.pdf,3,https://studentleadership.seattlecentral.edu/s...,Leadership & Orientation Training,Leadership & Orientation Training,same,2022-23,allocation,12600.00,"$12,600.00",
8,2023-24_sa_fee_memo.pdf,3,https://studentleadership.seattlecentral.edu/s...,Learning Support Network,Learning Support Network,same,2022-23,allocation,419159.00,"$419,159.00",
9,2023-24_sa_fee_memo.pdf,3,https://studentleadership.seattlecentral.edu/s...,M. Rosetta Hunter Art Gallary,M. Rosetta Hunter Art Gallery,spelling_variant,2022-23,allocation,57513.12,"$57,513.12",


In [35]:
# ============================================================
# CANONICAL DATASET VALIDATION
# ============================================================

print("Validating canonical dataset...\n")

# 1. Expected row count
assert len(sa_budget_canonical) == 155

print("✓ Canonical row count = 155")


# 2. No duplicate program/year/metric combinations
duplicate_keys = sa_budget_canonical[
    sa_budget_canonical.duplicated(
        subset=[
            "program_name_standardized",
            "fiscal_year",
            "metric_type",
        ],
        keep=False,
    )
]

assert duplicate_keys.empty, (
    "Duplicate canonical rows found."
)

print("✓ No duplicate program/year/metric combinations")


# 3. Confirm expected source by year/metric
source_check = (
    sa_budget_canonical
    .groupby(
        [
            "fiscal_year",
            "metric_type",
            "source_document",
        ]
    )
    .size()
    .reset_index(name="rows")
)

display(source_check)


# 4. Confirm allocation totals
allocation_totals = (
    sa_budget_canonical[
        sa_budget_canonical["metric_type"]
        == "allocation"
    ]
    .groupby("fiscal_year")["amount"]
    .sum()
)

expected_allocation_totals = {
    "2022-23": 1_500_000.00,
    "2023-24": 1_513_171.18,
    "2024-25": 1_733_008.51,
    "2025-26": 1_801_888.48,
}

for year, expected in expected_allocation_totals.items():
    actual = allocation_totals.loc[year]

    assert abs(actual - expected) < 0.01, (
        f"{year} allocation failed: "
        f"${actual:,.2f}"
    )

    print(
        f"✓ {year} allocation: "
        f"${actual:,.2f}"
    )


# 5. Confirm request totals
request_totals = (
    sa_budget_canonical[
        sa_budget_canonical["metric_type"]
        == "request"
    ]
    .groupby("fiscal_year")["amount"]
    .sum()
)

expected_request_totals = {
    "2023-24": 1_633_326.63,
    "2024-25": 1_502_598.56,
    "2025-26": 2_046_910.19,
}

for year, expected in expected_request_totals.items():
    actual = request_totals.loc[year]

    assert abs(actual - expected) < 0.01, (
        f"{year} request failed: "
        f"${actual:,.2f}"
    )

    print(
        f"✓ {year} request: "
        f"${actual:,.2f}"
    )


# 6. Ensure biennial request context survived
biennial_rows = sa_budget_canonical[
    (
        sa_budget_canonical["fiscal_year"]
        == "2024-25"
    )
    &
    (
        sa_budget_canonical["metric_type"]
        == "request"
    )
]

assert (
    biennial_rows["metric_context"]
    == "biennial_adjustment_request"
).all()

print("✓ FY2024-25 request context preserved")


print("\n" + "=" * 55)
print("CANONICAL DATASET VALIDATION PASSED ✓")
print("=" * 55)

Validating canonical dataset...

✓ Canonical row count = 155
✓ No duplicate program/year/metric combinations


,fiscal_year,metric_type,source_document,rows
0,2022-23,allocation,2023-24_sa_fee_memo.pdf,20
1,2023-24,allocation,2025-26_sa_budget_summary.pdf,24
2,2023-24,request,2023-24_sa_fee_memo.pdf,20
3,2024-25,allocation,2025-26_sa_budget_summary.pdf,24
4,2024-25,request,2024-25_sa_fee_memo.pdf,19
5,2025-26,allocation,2025-26_sa_budget_summary.pdf,24
6,2025-26,request,2025-26_sa_budget_summary.pdf,24


✓ 2022-23 allocation: $1,500,000.00
✓ 2023-24 allocation: $1,513,171.18
✓ 2024-25 allocation: $1,733,008.51
✓ 2025-26 allocation: $1,801,888.48
✓ 2023-24 request: $1,633,326.63
✓ 2024-25 request: $1,502,598.56
✓ 2025-26 request: $2,046,910.19
✓ FY2024-25 request context preserved

CANONICAL DATASET VALIDATION PASSED ✓


In [36]:
check_canonical = pd.read_csv(
    canonical_path
)

print("Rows:", len(check_canonical))
print("Columns:", len(check_canonical.columns))

NameError: name 'canonical_path' is not defined